In [1]:
import numpy as np
import pandas as pd

In [29]:
df=pd.read_csv('Titanic-Dataset.csv')
df.shape

(891, 12)

In [31]:
def memory_optimization(df):

     # MEMORY BEFORE OPTIMIZATION

    start_memory = df.memory_usage(deep=True).sum()/1024**2

    print(f"Memory Before Optimization : {start_memory:.2f} Mb")

    # Optimizing the memory

    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_integer_dtype(col_type):
            cmin=df[col].min()
            cmax=df[col].max()


            if cmin >= np.iinfo(np.int8).min and cmax <= np.iinfo(np.int8).max:
                df[col]=df[col].astype(np.int8)
            elif cmin >= np.iinfo(np.int16).min and cmax <= np.iinfo(np.int16).max:
                df[col]=df[col].astype(np.int16)
            elif cmin >= np.iinfo(np.int32).min and cmax <= np.iinfo(np.int32).max:
                df[col]=df[col].astype(np.int32)
            else:
                df[col]=df[col].astype(np.int64)

        elif pd.api.types.is_float_dtype(col_type):

            cmin = df[col].min()
            cmax = df[col].max()

            if cmin >= np.finfo(np.float16).min and cmax <= np.finfo(np.float16).max:
                df[col] = df[col].astype(np.float16)
            elif cmin >= np.finfo(np.float32).min and cmax <= np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)

    end_memory = df.memory_usage(deep=True).sum()/1024**2

    print(f"Memory after optimization : {end_memory:.2f} Mb")

    per = ((start_memory-end_memory)/start_memory)*100

    print(f"Memory Reduced By      : {per:.2f}%")

    return df

In [33]:
op_df=memory_optimization(df)

Memory Before Optimization : 0.28 Mb
Memory after optimization : 0.24 Mb
Memory Reduced By      : 14.01%


In [35]:
op_df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250000,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.312500,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.925781,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.093750,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.046875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.000000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.000000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.453125,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.000000,C148,C


In [41]:
def null_value_treatment(df,threshold=40):
    null_before = pd.DataFrame(
        {
            'Null_Count' : df.isna().sum(),
            'Null_Percentage' : (df.isna().sum()/len(df))*100,
            'Dtype' : df.dtypes
        }
    )

    print(f"Before Treatment : Null Values \n{null_before}")

    # Null value treatment

    dropped_col=[]
    filled_col=[]

    for col in df.columns:

        null_per = (df[col].isna().sum()/len(df[col]))*100

        if null_per > threshold:
            dropped_col.append(col)

        else:
            # if numeric cols
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(df[col].median())

                filled_col.append((col,"median"))
            else:
                df[col] = df[col].fillna(df[col].mode()[0])

                filled_col.append((col,"Mode"))

    df.drop(columns=dropped_col,inplace=True)

    print(f"Dropped Columns\n{dropped_col}")

    for col,method in filled_col:
        print(f"{col} filled using --> {method}")

    null_after = pd.DataFrame({
        'Null_Count' : df.isna().sum(),
        'Null_percentage' : (df.isna().sum()/len(df))*100,
        'DTypes' : df.dtypes
    })

    print(f"After Treatment : Null Values \n{null_after}")

    return df

In [43]:
no_null_df=null_value_treatment(op_df,30)

Before Treatment : Null Values 
             Null_Count  Null_Percentage    Dtype
PassengerId           0         0.000000    int16
Survived              0         0.000000     int8
Pclass                0         0.000000     int8
Name                  0         0.000000   object
Sex                   0         0.000000   object
Age                 177        19.865320  float16
SibSp                 0         0.000000     int8
Parch                 0         0.000000     int8
Ticket                0         0.000000   object
Fare                  0         0.000000  float16
Cabin               687        77.104377   object
Embarked              2         0.224467   object
Dropped Columns
['Cabin']
PassengerId filled using --> median
Survived filled using --> median
Pclass filled using --> median
Name filled using --> Mode
Sex filled using --> Mode
Age filled using --> median
SibSp filled using --> median
Parch filled using --> median
Ticket filled using --> Mode
Fare filled using --> 

In [47]:
no_null_df.isna().sum().sum()

0